# 09 — Likelihood Ratio Tests
**References:** Casella & Berger (2002) Ch. 8 · Lehmann & Romano (2005) Ch. 12 · Wasserman (2004) Ch. 10

## Narrative thread
```
Generalized LRT -> Wilks theorem -> Wald test -> Score test -> Asymptotic equivalence -> Model comparison
```

## 1. The generalized likelihood ratio test

For $H_0: \theta \in \Theta_0$ vs $H_1: \theta \in \Theta_1 = \Theta \setminus \Theta_0$, the **likelihood ratio statistic** is:

$$\Lambda(\mathbf{x}) = \frac{\sup_{\theta \in \Theta_0} L(\theta \mid \mathbf{x})}{\sup_{\theta \in \Theta} L(\theta \mid \mathbf{x})} = \frac{L(\hat{\theta}_0)}{L(\hat{\theta}_{\text{MLE}})} \in [0,1]$$

The **LRT** rejects $H_0$ when $\Lambda(\mathbf{x}) \leq k$ (equivalently, when $-2\log\Lambda$ is large).

**Wilks' Theorem:** Under $H_0$ and regularity conditions, as $n \to \infty$:
$$-2\log\Lambda(\mathbf{X}) \xrightarrow{d} \chi^2(r)$$
where $r = \dim(\Theta) - \dim(\Theta_0)$ is the number of constraints imposed by $H_0$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# ── Wilks theorem verification for Normal mean test ────────────────────────
# H0: mu = mu0  vs H1: mu != mu0
# Under H0, MLE under restriction: mu_hat_0 = mu0
# Under H1, MLE: mu_hat = x_bar
# LRT stat: n*(x_bar - mu0)^2 / sigma^2  ~ chi^2(1) under H0

mu0    = 0.0
sigma  = 1.0
ns_lrt = [10, 30, 100]
n_reps = 30_000

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
x_chi2 = np.linspace(0, 15, 300)

for ax, n in zip(axes, ns_lrt):
    lrt_stats = []
    for _ in range(n_reps):
        x    = np.random.normal(mu0, sigma, n)
        xb   = x.mean()
        # -2 log Lambda = n*(x_bar - mu0)^2 / sigma^2
        lrt_stats.append(n * (xb - mu0)**2 / sigma**2)

    lrt_stats = np.array(lrt_stats)
    ax.hist(lrt_stats, bins=80, density=True, color='#4361ee', alpha=0.7, edgecolor='white')
    ax.plot(x_chi2, stats.chi2.pdf(x_chi2, df=1), 'r-', lw=2.5, label='$\\chi^2(1)$')
    ax.set_xlim(0, 12)
    ax.set_title(f'n={n}: LRT stat $-2\\log\\Lambda$\n'
                  f'KS p={stats.kstest(lrt_stats, lambda x: stats.chi2.cdf(x, 1)).pvalue:.3f} (>0.05 = good fit)')
    ax.set_xlabel('LRT statistic'); ax.legend()

plt.suptitle("Wilks' Theorem: $-2\\log\\Lambda \\xrightarrow{d} \\chi^2(1)$ under $H_0: \\mu=0$",
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Three asymptotically equivalent tests

For $H_0: \theta = \theta_0$ vs $H_1: \theta \neq \theta_0$ (scalar $\theta$), three tests are each $\chi^2(1)$ asymptotically:

| Test | Statistic | Formula | When convenient |
|---|---|---|---|
| **Wald** | $W$ | $(\hat\theta - \theta_0)^2 / \widehat{\text{Var}}(\hat\theta)$ | When MLE is easy |
| **Score (Rao)** | $S$ | $[\ell'(\theta_0)]^2 / I_n(\theta_0)$ | When fitting $H_0$ model is easier |
| **LRT (Wilks)** | $G^2$ | $2[\ell(\hat\theta) - \ell(\theta_0)]$ | Best finite-sample performance |

**Ordering in small samples:** $W \geq G^2 \geq S$ typically — Wald is most conservative, Score most liberal. LRT is generally preferred.

**Multiparameter extension:** For $H_0: \boldsymbol{h}(\boldsymbol{\theta}) = \mathbf{0}$ ($r$ constraints):
$$W = n(\hat{\boldsymbol{\theta}} - \boldsymbol{\theta}_0)^\top \hat{I}(\hat{\boldsymbol{\theta}})\,(\hat{\boldsymbol{\theta}} - \boldsymbol{\theta}_0) \xrightarrow{d} \chi^2(r)$$

In [ ]:
# ── Compare Wald, Score, LRT for Poisson mean ─────────────────────────────
# X1,...,Xn ~ Poisson(lambda),  H0: lambda = lambda0
# MLE: lambda_hat = x_bar
# Fisher info: I(lambda) = 1/lambda

lambda0 = 2.0
n_small = 15  # small n to see differences
n_reps  = 30_000

wald_stats  = []
score_stats = []
lrt_stats   = []

for _ in range(n_reps):
    x      = np.random.poisson(lambda0, n_small)
    lhat   = x.mean()  # MLE
    if lhat < 1e-10: continue

    # Wald: (lambda_hat - lambda0)^2 / Var(lambda_hat) = n*(lambda_hat-lambda0)^2 / lambda_hat
    W = n_small * (lhat - lambda0)**2 / lhat

    # Score: [sum(xi - lambda0)]^2 / (n * lambda0) = n*(x_bar - lambda0)^2 / lambda0
    S = n_small * (lhat - lambda0)**2 / lambda0

    # LRT: 2 * [sum(xi*log(xi/lambda0) - (xi - lambda0))]
    # = 2 * [n*lhat*log(lhat/lambda0) - n*(lhat - lambda0)]
    G2 = 2 * n_small * (lhat * np.log(lhat / lambda0) - (lhat - lambda0))

    wald_stats.append(W)
    score_stats.append(S)
    lrt_stats.append(G2)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
x_chi2 = np.linspace(0, 12, 300)

for ax, (stats_arr, name, c) in zip(axes, [
    (wald_stats,  'Wald (W)', '#4361ee'),
    (score_stats, 'Score (S)', '#06d6a0'),
    (lrt_stats,   'LRT ($G^2$)', '#f72585'),
]):
    arr = np.array(stats_arr)
    ks_p = stats.kstest(arr, lambda x: stats.chi2.cdf(x, 1)).pvalue
    ax.hist(arr, bins=80, density=True, color=c, alpha=0.7, range=(0,12), edgecolor='white')
    ax.plot(x_chi2, stats.chi2.pdf(x_chi2, 1), 'k-', lw=2, label='$\\chi^2(1)$')
    ax.set_title(f'{name}\nKS test p={ks_p:.4f}\nSize-0.05 rejection rate: {(arr > stats.chi2.ppf(0.95,1)).mean():.3f}')
    ax.set_xlabel('Test statistic'); ax.legend(fontsize=8)

plt.suptitle(f'Wald vs Score vs LRT — Poisson, n={n_small} (small n shows differences)\n'
              'LRT typically closest to χ²(1); Wald can have poor small-sample calibration',
              fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. LRT for composite hypotheses: goodness-of-fit and model comparison

**Goodness-of-fit (Pearson chi-squared):** Test $H_0: \mathbf{p} = \mathbf{p}_0$ for multinomial data:
$$X^2 = \sum_{j=1}^k \frac{(O_j - E_j)^2}{E_j} \xrightarrow{d} \chi^2(k-1)$$

The LRT version: $G^2 = 2\sum_j O_j \log(O_j/E_j) \xrightarrow{d} \chi^2(k-1)$ — the **G-test**.

**Nested model comparison:** For models $M_0 \subset M_1$ with $r$ extra parameters:
$$G^2 = 2[\ell(\hat{\theta}_1) - \ell(\hat{\theta}_0)] \xrightarrow{d} \chi^2(r)$$

This is the foundation of ANOVA, logistic regression testing, and GLM model selection.

## 4. Key takeaways

| Test | Statistic | Distribution under $H_0$ | Key advantage |
|---|---|---|---|
| LRT | $2[\ell(\hat\theta) - \ell(\theta_0)]$ | $\chi^2(r)$ | Best finite-sample calibration |
| Wald | $(\hat\theta - \theta_0)^\top \hat I (\hat\theta - \theta_0)$ | $\chi^2(r)$ | Simple; plug-in |
| Score | $s(\theta_0)^\top I^{-1}(\theta_0) s(\theta_0)$ | $\chi^2(r)$ | No full MLE needed |
| Pearson $X^2$ | $\sum (O-E)^2/E$ | $\chi^2(k-1)$ | Multinomial GoF |

**Next:** notebook 10 — asymptotic theory: delta method, influence functions, and semiparametric efficiency.